In [1]:
!pip install -q imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 23.9 MB/s eta 0:00:00


In [2]:
import os
import zipfile
import random
import numpy as np
import tensorflow as tf
import cv2
from tqdm import tqdm
from PIL import Image
import imagehash

from collections import defaultdict, Counter
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
zip_path = "/content/drive/MyDrive/Parkinson's Dataset Final.zip"
extract_path = "/content/dataset"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

BASE = "/content/dataset/Parkinson's Dataset Final"

print("Extracted:", os.listdir(BASE))

Extracted: ['New hand pd Dataset', 'Hand Pd']


In [6]:
HANDPD_ROOT = os.path.join(BASE, "Hand Pd")

SPIRAL_ROOT = os.path.join(HANDPD_ROOT, "Spiral_HandPD")
MEANDER_ROOT = os.path.join(HANDPD_ROOT, "Meander_HandPD")

image_paths = []
labels = []

def collect_handpd(root):

    for cls in os.listdir(root):

        cls_path = os.path.join(root, cls)

        if not os.path.isdir(cls_path):
            continue

        if cls.lower() == "healthy":
            lab = 0
        elif cls.lower() == "parkinson":
            lab = 1
        else:
            continue

        for r, d, files in os.walk(cls_path):
            for f in files:

                if f.lower().endswith((".png",".jpg",".jpeg",".bmp")):

                    image_paths.append(os.path.join(r,f))
                    labels.append(lab)

collect_handpd(SPIRAL_ROOT)
collect_handpd(MEANDER_ROOT)

print("Total images:",len(image_paths))

Total images: 808


In [7]:
def get_phash(img_path):

    try:

        img = Image.open(img_path).convert("L").resize((224,224))

        return imagehash.phash(img)

    except:
        return None


hash_to_paths = defaultdict(list)

for p in tqdm(image_paths):

    h = get_phash(p)

    if h is not None:

        hash_to_paths[str(h)].append(p)


final_keep = []

for h in hash_to_paths:
    final_keep.append(hash_to_paths[h][0])


path_to_label = dict(zip(image_paths,labels))

clean_paths=[]
clean_labels=[]

for p in final_keep:

    clean_paths.append(p)
    clean_labels.append(path_to_label[p])

print("Clean dataset:",len(clean_paths))

100%|██████████| 808/808 [00:08<00:00, 90.55it/s] 

Clean dataset: 735


In [8]:
paths=np.array(clean_paths)
labels=np.array(clean_labels)

train_paths,test_paths,y_train,y_test=train_test_split(
paths,
labels,
test_size=0.2,
stratify=labels,
random_state=42
)

val_paths,test_paths,y_val,y_test=train_test_split(
test_paths,
y_test,
test_size=0.5,
stratify=y_test,
random_state=42
)

print("Train:",Counter(y_train))
print("Val:",Counter(y_val))
print("Test:",Counter(y_test))

Train: Counter({np.int64(1): 473, np.int64(0): 115})
Val: Counter({np.int64(1): 59, np.int64(0): 14})
Test: Counter({np.int64(1): 59, np.int64(0): 15})


In [9]:
IMG_SIZE=224
BATCH_SIZE=16

def preprocess(path):

    path=path.numpy().decode()

    img=cv2.imread(path,cv2.IMREAD_GRAYSCALE)

    img=cv2.resize(img,(IMG_SIZE,IMG_SIZE))

    clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))

    img=clahe.apply(img)

    img=img.astype(np.float32)/255.0

    img=np.expand_dims(img,-1)

    return img


def tf_preprocess(path,label):

    img=tf.py_function(preprocess,[path],tf.float32)

    img.set_shape((IMG_SIZE,IMG_SIZE,1))

    return img,label

In [10]:
from tensorflow.keras import layers

data_augmentation=tf.keras.Sequential([

layers.RandomFlip("horizontal"),

layers.RandomContrast(0.1)

])

In [11]:
def make_train_ds(paths,labels):

    ds=tf.data.Dataset.from_tensor_slices((paths,labels))

    ds=ds.shuffle(2000)

    ds=ds.map(tf_preprocess,num_parallel_calls=tf.data.AUTOTUNE)

    ds=ds.batch(BATCH_SIZE)

    ds=ds.map(lambda x,y:(data_augmentation(x,training=True),y))

    ds=ds.prefetch(tf.data.AUTOTUNE)

    return ds


def make_eval_ds(paths,labels):

    ds=tf.data.Dataset.from_tensor_slices((paths,labels))

    ds=ds.map(tf_preprocess,num_parallel_calls=tf.data.AUTOTUNE)

    ds=ds.batch(BATCH_SIZE)

    ds=ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds=make_train_ds(train_paths,y_train)
val_ds=make_eval_ds(val_paths,y_val)
test_ds=make_eval_ds(test_paths,y_test)

In [12]:
TARGET=1000

healthy=train_paths[y_train==0]
parkinson=train_paths[y_train==1]

def oversample(paths,target):

    paths=list(paths)

    extra=random.choices(paths,k=target-len(paths))

    return paths+extra


healthy_balanced=oversample(healthy,TARGET)
parkinson_balanced=oversample(parkinson,TARGET)

train_paths_balanced=np.array(healthy_balanced+parkinson_balanced)

y_train_balanced=np.array([0]*TARGET+[1]*TARGET)

train_ds=make_train_ds(train_paths_balanced,y_train_balanced)

In [13]:
PATCH_SIZE=16
EMBED_DIM=512
NUM_PATCHES=(IMG_SIZE//PATCH_SIZE)**2


class PatchEmbedding(layers.Layer):

    def __init__(self,patch_size,embed_dim):

        super().__init__()

        self.proj=layers.Conv2D(embed_dim,patch_size,patch_size)

    def call(self,x):

        x=self.proj(x)

        x=tf.reshape(x,[tf.shape(x)[0],-1,EMBED_DIM])

        return x

In [18]:
class AddPositionEmbedding(layers.Layer):

    def __init__(self, num_patches, embed_dim):
        super().__init__()

        self.num_patches = num_patches
        self.embed_dim = embed_dim

    def build(self, input_shape):

        self.pos_embed = self.add_weight(
            shape=(1, self.num_patches + 1, self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

        self.cls_token = self.add_weight(
            shape=(1,1,self.embed_dim),
            initializer="random_normal",
            trainable=True
        )

    def call(self, x):

        batch = tf.shape(x)[0]

        cls = tf.repeat(self.cls_token, repeats=batch, axis=0)

        x = tf.concat([cls, x], axis=1)

        return x + self.pos_embed

In [19]:
class TransformerEncoder(layers.Layer):

    def __init__(self,embed_dim,num_heads,mlp_dim):

        super().__init__()

        self.norm1=layers.LayerNormalization()

        self.attn=layers.MultiHeadAttention(num_heads=num_heads,key_dim=embed_dim//num_heads)

        self.norm2=layers.LayerNormalization()

        self.mlp=tf.keras.Sequential([
            layers.Dense(mlp_dim,activation="gelu"),
            layers.Dense(embed_dim)
        ])

    def call(self,x):

        x=x+self.attn(self.norm1(x),self.norm1(x))

        x=x+self.mlp(self.norm2(x))

        return x

In [23]:
def build_model():

    inputs = layers.Input((224,224,1))

    # ---------------- CNN Feature Extractor ----------------

    x = layers.Conv2D(32,3,padding="same",activation="relu")(inputs)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(64,3,padding="same",activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    x = layers.Conv2D(128,3,padding="same",activation="relu")(x)
    x = layers.MaxPooling2D()(x)

    # ---------------- Patch Embedding ----------------

    x = layers.Conv2D(EMBED_DIM, PATCH_SIZE, PATCH_SIZE)(x)

    x = tf.reshape(x, [tf.shape(x)[0], -1, EMBED_DIM])

    x = AddPositionEmbedding(NUM_PATCHES, EMBED_DIM)(x)

    # ---------------- Transformer ----------------

    for _ in range(4):

        x = TransformerEncoder(EMBED_DIM, 8, 1024)(x)

    x = layers.LayerNormalization()(x)

    x = x[:,0]

    # ---------------- Dense Classifier ----------------

    x = layers.Dense(512,activation="relu")(x)

    x = layers.Dense(64,activation="relu")(x)
    x = layers.Dense(32,activation="relu")(x)
    x = layers.Dense(16,activation="relu")(x)
    x = layers.Dense(8,activation="relu")(x)
    x = layers.Dense(4,activation="relu")(x)

    outputs = layers.Dense(2,activation="softmax")(x)

    model = tf.keras.Model(inputs,outputs)

    return model


model = build_model()

model.summary()

ValueError: A KerasTensor cannot be used as input to a TensorFlow function. A KerasTensor is a symbolic placeholder for a shape and dtype, used when constructing Keras Functional models or Keras Functions. You can only use it as input to a Keras layer or a Keras operation (from the namespaces `keras.layers` and `keras.ops`). You are likely doing something like:

```
x = Input(...)
...
tf_fn(x)  # Invalid.
```

What you should do instead is wrap `tf_fn` in a layer:

```
class MyLayer(Layer):
    def call(self, x):
        return tf_fn(x)

x = MyLayer()(x)
```


In [21]:
model.compile(

optimizer=tf.keras.optimizers.AdamW(
learning_rate=1e-4,
weight_decay=1e-4
),

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]
)

In [22]:
history=model.fit(

train_ds,

validation_data=val_ds,

epochs=60
)

Epoch 1/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 245ms/step - accuracy: 0.4785 - loss: 0.6987 - val_accuracy: 0.8082 - val_loss: 0.6917
Epoch 2/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 27s 212ms/step - accuracy: 0.4884 - loss: 0.6932 - val_accuracy: 0.8082 - val_loss: 0.6917
Epoch 3/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 198ms/step - accuracy: 0.4976 - loss: 0.6932 - val_accuracy: 0.8082 - val_loss: 0.6919
Epoch 4/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 196ms/step - accuracy: 0.5156 - loss: 0.6931 - val_accuracy: 0.8082 - val_loss: 0.6918
Epoch 5/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 199ms/step - accuracy: 0.4930 - loss: 0.6932 - val_accuracy: 0.8082 - val_loss: 0.6919
Epoch 6/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 197ms/step - accuracy: 0.5035 - loss: 0.6931 - val_accuracy: 0.8082 - val_loss: 0.6921
Epoch 7/60
125/125 ━━━━━━━━━━━━━━━━━━━━ 25s 197ms/step - accuracy: 0.4848 - loss: 0.6932 - val_accuracy: 0.8082 - val_loss: 0.6920
Epoch 8/60


KeyboardInterrupt: 